# Tape Preview — Canny + HoughLinesP

Camera has chromatic aberration (pink fringing) and lens shading.  
Avoid color. Detect tape as **line segments** via edge detection → robust to lighting.

Pipeline: ROI crop (bottom) → grayscale → blur → Canny → HoughLinesP → filter by angle → pick innermost line each side → midpoint.

View: **frame + detection** (left) + **edge map** (right).

In [1]:
import time
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display

def gstreamer_pipeline(width=640, height=480, fps=30):
    return (
        f"nvarguscamerasrc ! "
        f"video/x-raw(memory:NVMM), width={width}, height={height}, "
        f"format=NV12, framerate={fps}/1 ! "
        f"nvvidconv flip-method=0 ! "
        f"video/x-raw, width={width}, height={height}, format=BGRx ! "
        f"videoconvert ! video/x-raw, format=BGR ! appsink"
    )

cap = cv2.VideoCapture(gstreamer_pipeline(), cv2.CAP_GSTREAMER)
assert cap.isOpened(), "Camera failed to open!"

image_widget = widgets.Image(format='jpeg', width=1280, height=480)

wide = widgets.Layout(width='500px')
style = {'description_width': '120px'}
canny_lo     = widgets.IntSlider(value=50,  min=0,  max=255, description='canny lo',    layout=wide, style=style, continuous_update=True)
canny_hi     = widgets.IntSlider(value=150, min=0,  max=500, description='canny hi',    layout=wide, style=style, continuous_update=True)
blur         = widgets.IntSlider(value=15,   min=1,  max=21,  step=2, description='blur k',      layout=wide, style=style, continuous_update=True)
min_line_len = widgets.IntSlider(value=40,  min=5,  max=300, description='min line len', layout=wide, style=style, continuous_update=True)
max_line_gap = widgets.IntSlider(value=10,  min=0,  max=100, description='max line gap', layout=wide, style=style, continuous_update=True)
min_angle    = widgets.IntSlider(value=20,  min=0,  max=89,  description='min angle deg', layout=wide, style=style, continuous_update=True)
roi_pct      = widgets.IntSlider(value=40,  min=20, max=100, description='roi % (bottom)', layout=wide, style=style, continuous_update=True)
live = widgets.Label(value='live values: --')

display(image_widget)
display(widgets.VBox([canny_lo, canny_hi, blur, min_line_len, max_line_gap, min_angle, roi_pct, live]))

def bgr_to_jpeg(frame):
    _, buf = cv2.imencode('.jpeg', frame)
    return bytes(buf)

print("Camera ready.")

Image(value=b'', format='jpeg', height='480', width='1280')

Camera ready.


In [ ]:
from IPython import get_ipython
kernel = get_ipython().kernel

EMA_ALPHA = 0.35      # smoothing (higher = snappier, lower = steadier)
HOLD_FRAMES = 15      # hold last EMA when side has no data

def fit_side_linear(side_segs):
    """Deg-1 fit x=m*y+b with 2-sigma outlier rejection. Return (m,b,y_min,y_max) or None."""
    if len(side_segs) < 1:
        return None
    xs, ys = [], []
    for _, (x1, y1, x2, y2) in side_segs:
        xs.extend([x1, x2])
        ys.extend([y1, y2])
    xs_arr = np.asarray(xs, dtype=np.float64)
    ys_arr = np.asarray(ys, dtype=np.float64)
    if ys_arr.ptp() < 2 or len(xs_arr) < 2:
        return None
    m, b = np.polyfit(ys_arr, xs_arr, 1)
    resid = xs_arr - (m * ys_arr + b)
    sigma = float(np.std(resid))
    if sigma > 1.0 and len(xs_arr) >= 4:
        mask = np.abs(resid) < 2.0 * sigma
        if mask.sum() >= 2 and np.ptp(ys_arr[mask]) > 1:
            m, b = np.polyfit(ys_arr[mask], xs_arr[mask], 1)
            xs_arr, ys_arr = xs_arr[mask], ys_arr[mask]
    return float(m), float(b), float(ys_arr.min()), float(ys_arr.max())

def polyline(vis, xs, ys, color, thickness):
    pts = np.stack([xs, ys], axis=1).astype(np.int32)
    cv2.polylines(vis, [pts], isClosed=False, color=color, thickness=thickness)

# Persistent state across frames
ema = {'l': None, 'r': None}          # each: (m, b)
last_data_y = {'l': None, 'r': None}  # each: (y_min, y_max)
stale = {'l': 0, 'r': 0}

try:
    while True:
        kernel.do_one_iteration()

        ret, frame = cap.read()
        if not ret:
            time.sleep(0.01)
            continue

        h, w = frame.shape[:2]
        center_x = w // 2
        bot_pt = (center_x, h - 1)

        clo_v  = canny_lo.value
        chi_v  = canny_hi.value
        k_v    = blur.value if blur.value % 2 == 1 else blur.value + 1
        mll_v  = min_line_len.value
        mlg_v  = max_line_gap.value
        ang_v  = min_angle.value
        roi_v  = roi_pct.value
        live.value = (f'live values: canny={clo_v}/{chi_v} blur={k_v} '
                      f'minLen={mll_v} maxGap={mlg_v} minAng={ang_v} roi={roi_v}%')

        roi_h  = max(1, int(h * roi_v / 100))
        roi_y0 = h - roi_h

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (k_v, k_v), 0)
        roi_gray = gray[roi_y0:, :]

        edges = cv2.Canny(roi_gray, clo_v, chi_v)

        lines = cv2.HoughLinesP(
            edges, rho=1, theta=np.pi / 180, threshold=40,
            minLineLength=mll_v, maxLineGap=mlg_v,
        )

        vis = frame.copy()
        cv2.line(vis, (center_x, 0), (center_x, h - 1), (0, 255, 255), 1)
        cv2.line(vis, (0, roi_y0), (w, roi_y0), (255, 255, 255), 1)

        kept = []
        if lines is not None:
            for seg in lines[:, 0, :]:
                x1, y1, x2, y2 = seg
                dx = x2 - x1
                dy = y2 - y1
                angle = abs(np.degrees(np.arctan2(dy, dx)))
                if angle > 90:
                    angle = 180 - angle
                if angle < ang_v:
                    continue
                mx = (x1 + x2) // 2
                y1f = y1 + roi_y0
                y2f = y2 + roi_y0
                kept.append((mx, (x1, y1f, x2, y2f)))

        buckets = {
            'l': [k for k in kept if k[0] <  center_x],
            'r': [k for k in kept if k[0] >= center_x],
        }

        fits = {}
        for side in ('l', 'r'):
            raw = fit_side_linear(buckets[side])
            if raw is None:
                stale[side] += 1
                if ema[side] is not None and stale[side] <= HOLD_FRAMES and last_data_y[side] is not None:
                    fits[side] = (ema[side][0], ema[side][1], last_data_y[side])
                else:
                    fits[side] = None
                continue
            m, b, y_min, y_max = raw
            if ema[side] is None:
                ema[side] = (m, b)
            else:
                em, eb = ema[side]
                ema[side] = ((1 - EMA_ALPHA) * em + EMA_ALPHA * m,
                             (1 - EMA_ALPHA) * eb + EMA_ALPHA * b)
            last_data_y[side] = (y_min, y_max)
            stale[side] = 0
            fits[side] = (ema[side][0], ema[side][1], last_data_y[side])

        lf, rf = fits.get('l'), fits.get('r')

        if lf is not None and rf is not None:
            lm, lb, (ly_min, ly_max) = lf
            rm, rb, (ry_min, ry_max) = rf

            ly = np.linspace(ly_min, ly_max, 20)
            lx = lm * ly + lb
            ry = np.linspace(ry_min, ry_max, 20)
            rx = rm * ry + rb
            polyline(vis, lx, ly, (255, 0, 0),   2)
            polyline(vis, rx, ry, (0, 128, 255), 2)

            mid_y = np.linspace(roi_y0, h - 1, 20)
            mid_x = ((lm * mid_y + lb) + (rm * mid_y + rb)) / 2.0
            polyline(vis, mid_x, mid_y, (0, 255, 0), 3)

            bot_mx = int(mid_x[-1])
            err = bot_mx - center_x
            look_mx = int(mid_x[0])
            look_my = int(mid_y[0])
            cv2.arrowedLine(vis, bot_pt, (look_mx, look_my), (0, 255, 0), 3, tipLength=0.08)
            held = []
            if stale['l'] > 0: held.append(f"Lhold{stale['l']}")
            if stale['r'] > 0: held.append(f"Rhold{stale['r']}")
            tag = ' ' + ' '.join(held) if held else ''
            cv2.putText(vis, f"err={err:+d}{tag}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        else:
            missing = []
            if lf is None: missing.append('NO LEFT')
            if rf is None: missing.append('NO RIGHT')
            cv2.putText(vis, ' '.join(missing),
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        edges_full = np.zeros((h, w), dtype=np.uint8)
        edges_full[roi_y0:, :] = edges
        edges_bgr = cv2.cvtColor(edges_full, cv2.COLOR_GRAY2BGR)
        cv2.line(edges_bgr, (0, roi_y0), (w, roi_y0), (255, 255, 255), 1)
        combined = np.hstack([vis, edges_bgr])
        image_widget.value = bgr_to_jpeg(combined)
        time.sleep(0.03)

except KeyboardInterrupt:
    print("Preview stopped.")

In [ ]:
cap.release()
print("Camera released.")